# Esplorazione Leg19 — emendamenti e resoconti

**Obiettivo**: valutare se vale la pena estendere la pipeline di estrazione di `senato-akn`
da `ddlpres` (già estratto) a tutte le tipologie documentali di Leg19.

**Campione**: ~20 file per tipologia, parsati con `senato_akn.parser.parse_xml`.

**Domande**:
1. Il parser esistente regge su tutte le tipologie?
2. Quali metadati specifici hanno emendamenti e resoconti?
3. Quanto variano per dimensione e struttura?
4. Che domande civiche possiamo rispondere con questi dati?

## 1. Setup

In [ ]:
import json, urllib.request, random
from pathlib import Path
from collections import Counter, defaultdict
import xml.etree.ElementTree as ET

import sys
sys.path.insert(0, str(Path.cwd()))
from senato_akn.parser import parse_xml, NS

REPO = Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent
print(f'Workspace: {REPO}')

## 2. Campionamento file per tipologia

In [ ]:
# Ottieni il tree di Leg19
api = "https://api.github.com/repos/SenatoDellaRepubblica/AkomaNtosoBulkData/git/trees"

# SHA di Leg19 (da master)
req = urllib.request.Request(f"{api}/master")
with urllib.request.urlopen(req) as r:
    root = json.loads(r.read())
leg19_sha = next(item['sha'] for item in root['tree'] if item['path'] == 'Leg19')
print(f"Leg19 SHA: {leg19_sha}")

# Tree ricorsivo
req = urllib.request.Request(f"{api}/{leg19_sha}?recursive=1")
with urllib.request.urlopen(req) as r:
    leg19 = json.loads(r.read())

# Raggruppa per tipologia
by_type = defaultdict(list)
for item in leg19['tree']:
    if not item['path'].endswith('.akn.xml'):
        continue
    parts = item['path'].split('/')
    if len(parts) >= 3:
        by_type[parts[1]].append(item)

print(f"\nFile XML per tipologia:")
SAMPLE = 15  # file per tipologia
samples = {}
for t in ['ddlpres', 'emend', 'emendc', 'resaula', 'sommcomm', 'ddlmess', 'ddlcomm']:
    files = by_type.get(t, [])
    random.seed(42)
    chosen = random.sample(files, min(SAMPLE, len(files)))
    samples[t] = [(item['path'], item['size']) for item in chosen]
    print(f"  {t:15s} {len(files):>6,} totali → campione {len(chosen)}")

print(f"\nCampione totale: {sum(len(v) for v in samples.values())} file")

## 3. Parsing del campione

In [ ]:
import time, statistics

RAW_BASE = "https://raw.githubusercontent.com/SenatoDellaRepubblica/AkomaNtosoBulkData/master"

results = []
errors = []

for tipologia, files in samples.items():
    for path, size in files:
        url = f"{RAW_BASE}/{path}"
        try:
            req = urllib.request.Request(url)
            with urllib.request.urlopen(req, timeout=30) as r:
                xml_bytes = r.read()
            parsed = parse_xml(xml_bytes, path=path, legislatura='Leg19')
            parsed['tipologia'] = tipologia
            parsed['file_size'] = size
            results.append(parsed)
        except Exception as e:
            errors.append({'path': path, 'error': str(e)})
        time.sleep(0.1)  # rate limit

print(f"Parsati: {len(results)} / {sum(len(v) for v in samples.values())}")
print(f"Errori: {len(errors)}")
if errors:
    for e in errors[:5]:
        print(f"  ERR {e['path']}: {e['error']}")

## 4. Struttura dei metadati FRBR

In [ ]:
# Campi FRBR disponibili
print("Campi estratti dal parser:")
for k in ['work_uri', 'expression_uri', 'work_date', 'expression_date',
          'doc_title', 'short_title', 'articles_count', 'paragraphs_count', 'text_len']:
    filled = sum(1 for r in results if r.get(k))
    print(f"  {k:25s}: {filled}/{len(results)} compilati")

print(f"\nEsempio work_uri: {results[0]['work_uri']}")
print(f"Esempio expression_uri: {results[0]['expression_uri']}")

In [ ]:
# Analisi degli URI per capire cosa contengono
# work_uri es: /senato/leg19/1299/senato/emend/emend.xml
print("Work URI per tipologia (primo esempio):")
for tipologia in ['ddlpres', 'emend', 'emendc', 'resaula', 'sommcomm', 'ddlmess', 'ddlcomm']:
    for r in results:
        if r['tipologia'] == tipologia and r['work_uri']:
            print(f"  {tipologia:15s} → {r['work_uri']}")
            break

## 5. Metadati specifici per tipologia

Cerchiamo nell'XML campi specifici per emendamenti (proponente, articolo, esito)
e resoconti (seduta, presidente, gruppi).

In [ ]:
# Ispeziona l'XML di un emendamento per trovare metadati proponente
for r in results:
    if r['tipologia'] in ('emend', 'emendc'):
        print(f"URL: {r['work_uri']}")
        print(f"Titolo: {r['doc_title'][:100]}")
        print(f"Lunghezza testo: {r['text_len']} caratteri")
        print(f"Articoli: {r['articles_count']}, Paragrafi: {r['paragraphs_count']}")
        print()
        break

In [ ]:
# XML grezzo di un emendamento — cerchiamo nodi proponente e voting
emend_xml = None
for r in results:
    if r['tipologia'] in ('emend', 'emendc'):
        url = f"{RAW_BASE}/{r['path']}"
        req = urllib.request.Request(url)
        with urllib.request.urlopen(req) as resp:
            emend_xml = resp.read().decode('utf-8')
        print(f"Prime 2000 caratteri XML di un emendamento:\n")
        print(emend_xml[:2000])
        break

In [ ]:
# XML grezzo di un resoconto d'aula
for r in results:
    if r['tipologia'] == 'resaula':
        url = f"{RAW_BASE}/{r['path']}"
        req = urllib.request.Request(url)
        with urllib.request.urlopen(req) as resp:
            aula_xml = resp.read().decode('utf-8')
        print(f"Prime 2000 caratteri XML di un resoconto d'Aula:\n")
        print(aula_xml[:2000])
        break

## 6. Confronto dimensioni per tipologia

In [ ]:
import pandas as pd

df = pd.DataFrame(results)

print(f"{'Tipologia':15s} | {'File':>5s} | {'Testo medio':>12s} | {'Articoli medio':>14s} | {'File medio':>11s}")
print('-' * 65)
for t in ['ddlpres', 'emend', 'emendc', 'resaula', 'sommcomm', 'ddlmess', 'ddlcomm']:
    subset = df[df['tipologia'] == t]
    if len(subset) == 0:
        continue
    txt = subset['text_len'].mean()
    art = subset['articles_count'].mean()
    fsz = subset['file_size'].mean()
    print(f"{t:15s} | {len(subset):>5d} | {txt:>10,.0f} chr | {art:>10,.1f} | {fsz:>8,.0f} B")

print(f"\nDim. totale campione: {df['file_size'].sum():,} bytes")
print(f"Dim. media file: {df['file_size'].mean():,.0f} bytes")

## 7. Qualità metadati FRBR

In [ ]:
print(f"{'Campo':25s} | {'Compilati %':>12s} | {'Esempio'}")
print('-' * 70)
for col in ['work_date', 'expression_date', 'doc_title', 'short_title']:
    filled = df[col].notna().sum()
    example = df[df[col].notna()][col].iloc[0] if filled > 0 else '—'
    print(f"{col:25s} | {filled/len(df)*100:>5.1f}%       | {str(example)[:50]}")

## 8. Verdetto

Basandosi sui dati raccolti:

1. **Il parser regge?** Il parse ha funzionato su tutte le tipologie?
2. **Metadati emendamenti**: c'è proponente, articolo, esito nell'XML?
3. **Metadati resoconti**: c'è seduta, presidente, interventi?
4. **Volume**: la dimensione media dei file giustifica l'estrazione?
5. **Valore civico**: che domande possiamo rispondere con emendamenti e resoconti?

**Conclusioni**: conviene estendere la pipeline a tutte le tipologie? Se sì, cosa va modificato nel parser?